# A04: Metaprogramación, Descriptores y Metaclases

Nivel: **Avanzado** | Tema: Metaprogramación en Python

## Objetivos de Aprendizaje

Al finalizar este notebook, serás capaz de:

1. **Comprender** que las clases son objetos en Python y que `type` es la metaclase por defecto.
2. **Implementar** descriptores personalizados (data y non-data) para controlar acceso a atributos.
3. **Crear** metaclases que modifiquen el comportamiento de clases al momento de definirse.
4. **Utilizar** `__init_subclass__` como alternativa práctica a las metaclases para patrones como el registry.
5. **Aplicar** ABC y `abstractmethod` para forzar la implementación de interfaces.

## Analogía Central

Imagina una fábrica de automóviles:

- **Clase** = La fábrica que produce coches (cada `instancia` es un coche).
- **Metaclase** = La compañía que diseña y construye la fábrica misma. Decide cómo se fabricarán los coches.
- **Descriptor** = Un supervisor especializado asignado a cada pieza del coche. Verifica que el motor sea del tipo correcto, que las ruedas tengan el tamaño adecuado, etc. Interviene cada vez que accedes o modificas una parte del coche.

```ascii
  ┌──────────────────────────────────────────────────────────┐
  │                    METALEVEL                             │
  │                                                          │
  │   ┌──────────┐   type()    ┌──────────────┐             │
  │   │  type    │ ──────────► │ MiMetaclase  │             │
  │   └──────────┘             └──────┬───────┘             │
  │        │                          │ crea                │
  │        │ crea                     ▼                     │
  │        ▼                   ┌──────────────┐             │
  │   ┌──────────┐            │   MiClase    │             │
  │   │ Clase    │ ◄───────── └──────────────┘             │
  │   └────┬─────┘             tiene descriptores           │
  │        │ instancia                                       │
  │        ▼                                                 │
  │   ┌──────────┐                                          │
  │   │ Objeto   │  ◄── accede a atributo                   │
  │   └──────────┘      ──► descriptor intercepta           │
  │                                                          │
  └──────────────────────────────────────────────────────────┘
```

---
## 1. Clases como objetos

En Python **todo es un objeto**: enteros, cadenas, funciones… y también las clases. Esto significa que puedes pasar clases como argumentos, asignarlas a variables y hasta crearlas dinámicamente.

### `type()` dual

`type()` tiene dos usos:
- **Como función**: retorna el tipo de un objeto.
- **Como constructor**: crea una clase en tiempo de ejecución.

```ascii
   type(objeto)                 type(nombre, bases, dict)
       │                               │
       ▼                               ▼
  Retorna <class 'X'>          Retorna una nueva CLASE
  (inspecta)                    (construye)
```

In [ ]:
# type() como función inspectora
class Persona:
    def __init__(self, nombre: str, edad: int) -> None:
        self.nombre = nombre
        self.edad = edad

p = Persona("Ana", 30)

print(f"Tipo del objeto p:  {type(p)}")
print(f"Tipo de la clase:   {type(Persona)}")
print(f"¿Persona es instancia de type? {isinstance(Persona, type)}")

### Crear una clase dinámicamente con `type()`

Los tres argumentos de `type()` son:
1. **`nombre`**: nombre de la clase (str).
2. **`bases`**: tupla de clases padre.
3. **`dict`**: diccionario de atributos y métodos.

In [ ]:
# Crear clase dinámicamente
def saludar(self) -> str:
    return f"Hola, soy {self.nombre} y tengo {self.edad} años."

PersonaDinamica = type(
    "PersonaDinamica",          # nombre
    (object,),                  # bases (padres)
    {
        "__init__": lambda self, nombre, edad: setattr(self, "nombre", nombre) or setattr(self, "edad", edad),
        "saludar": saludar,
        "es_adulto": property(lambda self: self.edad >= 18),
    }
)

p2 = PersonaDinamica("Luis", 25)
print(p2.saludar())
print(f"Es adulto: {p2.es_adulto}")
print(f"Clase creada dinámicamente: {PersonaDinamica.__name__}")

### Clase `type` es la metaclase por defecto

Cada clase que defines es, en realidad, una instancia de `type`. Cuando escribes `class MiClase: ...`, Python internamente ejecuta `MiClase = type('MiClase', (), {...})`.

In [ ]:
# Verificación de que type es la metaclase por defecto
class Saludo:
    pass

print(f"metaclase de Saludo: {type(Saludo)}")
print(f"¿type es metaclase de str? {type(str)}")
print(f"¿type es metaclase de int? {type(int)}")
print(f"¿type es metaclase de list? {type(list)}")

# La cadena de tipos completa
print(f"\ntype(42) = {type(42)}")
print(f"type(type(42)) = {type(type(42))}")

---
## 2. Descriptores

Un descriptor es cualquier objeto que implemente al menos uno de estos métodos mágicos:
- `__get__(self, obj, objtype=None)` → intercepta **lecturas** del atributo.
- `__set__(self, obj, value)` → intercepta **asignaciones** al atributo.
- `__delete__(self, obj)` → intercepta **eliminaciones** del atributo.

```ascii
  Objeto.atributo          →  descriptor.__get__(self, objeto, type(objeto))
  Objeto.atributo = valor  →  descriptor.__set__(self, objeto, valor)
  del Objeto.atributo      →  descriptor.__delete__(self, objeto)
```

### Tipos de descriptores

| Tipo | Métodos implementados | Ejemplo |
|------|----------------------|----------|
| **Data descriptor** | `__get__` + `__set__` (y/o `__delete__`) | `property`, validadores |
| **Non-data descriptor** | Solo `__get__` | Métodos, `classmethod`, `staticmethod` |

**Regla de prioridad** (data descriptor gana):
```
  __dict__ del objeto     vs     Data Descriptor
  (atributo de instancia)       (atributo de clase)
               ────────────────────►  GANA el descriptor
```

In [ ]:
# Descriptor de solo lectura (non-data descriptor)
class SoloLectura:
    """Descriptor que solo permite lectura."""
    def __set_name__(self, owner, name) -> None:
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return f"Valor protegido de {self.name}"


class Configuracion:
    version = SoloLectura()
    modo = SoloLectura()


cfg = Configuracion()
print(cfg.version)
print(cfg.modo)
# cfg.version = "otro"  # AttributeError - __set__ no implementado

In [ ]:
# Comparación: data descriptor vs non-data descriptor
class DataDescriptor:
    """Data descriptor: tiene __get__ y __set__."""
    def __set_name__(self, owner, name) -> None:
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(f"_data_{self.name}", 0)

    def __set__(self, obj, value) -> None:
        print(f"  [DataDescriptor.__set__] {self.name} = {value}")
        obj.__dict__[f"_data_{self.name}"] = value


class ObjetoEjemplo:
    x = DataDescriptor()


o = ObjetoEjemplo()
print("Asignando x = 10:")
o.x = 10  # invoca DataDescriptor.__set__
print(f"Leyendo o.x: {o.x}")

# Intentar bypassar el descriptor escribiendo directo a __dict__
o.__dict__["x"] = 999
print(f"\nDespués de o.__dict__['x'] = 999:")
print(f"Leyendo o.x: {o.x}")  # ¡Sigue usando el descriptor! Data descriptor gana.
print(f"o.__dict__['x']: {o.__dict__.get('x', 'no existe')}")

---
## 3. Descriptores en la práctica

### `property` ES un descriptor

`property` es el descriptor de datos más usado en Python. Detrás de `@property`, `@x.setter` y `@x.deleter` hay un objeto que implementa `__get__`, `__set__` y `__delete__`.

In [ ]:
# property es un descriptor integrado
print(f"tipo de property: {type(property)}")
print(f"¿property tiene __get__? {hasattr(property, '__get__')}")
print(f"¿property tiene __set__? {hasattr(property, '__set__')}")
print(f"¿property tiene __delete__? {hasattr(property, '__delete__')}")

### Implementar `property` a mano

Para entender qué hace `property` por debajo, implementemos una versión simplificada.

In [ ]:
class MiProperty:
    """Implementación simplificada de property como descriptor."""

    def __init__(self, fget=None, fset=None, fdel=None, doc=None) -> None:
        self.fget = fget
        self.fset = fset
        self.fdel = fdel
        self.__doc__ = doc or (fget.__doc__ if fget else None)

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.fget is None:
            raise AttributeError("atributo no legible")
        return self.fget(obj)

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("atributo no asignable")
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError("atributo no eliminable")
        self.fdel(obj)

    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__)

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__)


class Temperatura:
    def __init__(self, celsius: float = 0.0) -> None:
        self._celsius = celsius

    @MiProperty
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, valor: float) -> None:
        if valor < -273.15:
            raise ValueError(f"Temperatura {valor}°C menor que el cero absoluto")
        self._celsius = valor

    @MiProperty
    def fahrenheit(self) -> float:
        return self._celsius * 9 / 5 + 32


t = Temperatura(100)
print(f"Celsius:    {t.celsius}°C")
print(f"Fahrenheit: {t.fahrenheit}°F")
print(f"Docstring:  {Temperatura.celsius.__doc__}")
t.celsius = 0
print(f"\nDespués de t.celsius = 0:")
print(f"Fahrenheit: {t.fahrenheit}°F")

### Validación de tipo con descriptor `TypeChecked`

Un descriptor que valida el tipo de dato al asignar un atributo.

In [ ]:
class TypeChecked:
    """Descriptor que valida el tipo al asignar."""

    def __init__(self, expected_type: type) -> None:
        self.expected_type = expected_type
        self.name = ""

    def __set_name__(self, owner, name) -> None:
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(f"_tc_{self.name}")

    def __set__(self, obj, value) -> None:
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"'{self.name}' debe ser {self.expected_type.__name__}, "
                f"se recibió {type(value).__name__}"
            )
        obj.__dict__[f"_tc_{self.name}"] = value


class Producto:
    nombre = TypeChecked(str)
    precio = TypeChecked(float)
    stock = TypeChecked(int)

    def __init__(self, nombre: str, precio: float, stock: int) -> None:
        self.nombre = nombre
        self.precio = precio
        self.stock = stock

    def __repr__(self) -> str:
        return f"Producto(nombre='{self.nombre}', precio={self.precio}, stock={self.stock})"


p = Producto("Laptop", 999.99, 50)
print(p)

# Asignación válida
p.precio = 1299.99
print(f"Nuevo precio: {p.precio}")

# Asignación inválida - lanza TypeError
try:
    p.precio = "mil"  # str, no float
except TypeError as e:
    print(f"\nError controlado: {e}")

---
## 4. `__set_name__`

Cuando defines un descriptor como atributo de clase, Python llama automáticamente a `__set_name__` con:
- `owner`: la clase propietaria.
- `name`: el nombre del atributo donde se instanció.

Esto evita repetir el nombre del atributo en el descriptor.

In [ ]:
class DescriptorAutoNombre:
    """Descriptor que conoce su propio nombre sin hardcodearlo."""

    def __set_name__(self, owner, name) -> None:
        self.public_name = name
        self.private_name = f"_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        value = getattr(obj, self.private_name, None)
        print(f"  [__get__] Leyendo '{self.public_name}' = {value}")
        return value

    def __set__(self, obj, value) -> None:
        print(f"  [__set__] Escribiendo '{self.public_name}' = {value}")
        setattr(obj, self.private_name, value)


class Servidor:
    host = DescriptorAutoNombre()
    puerto = DescriptorAutoNombre()
    protocolo = DescriptorAutoNombre()

    def __init__(self, host: str, puerto: int, protocolo: str = "HTTP") -> None:
        self.host = host
        self.puerto = puerto
        self.protocolo = protocolo


srv = Servidor("localhost", 8080, "HTTPS")
print(f"host: {srv.host}")
print(f"puerto: {srv.puerto}")
print(f"protocolo: {srv.protocolo}")

# Acceso directo al descriptor desde la clase
print(f"\nDescriptor host conoce su nombre: {Servidor.host.public_name}")

---
## 5. Metaclases - Creando clases

Una metaclase es la **clase de una clase**. Define cómo se crean y comportan las clases.

```ascii
  type
    │
    ├──► MiMetaclase (hereda de type)
    │         │
    │         ├──► MiClase_A (creada por MiMetaclase)
    │         └──► MiClase_B (creada por MiMetaclase)
    │
    └──► Clases normales (creadas por type)
```

### `__metaclass__` (Python pre-3 - no usar)

En Python 2 se usaba `__metaclass__` dentro del cuerpo de la clase. En Python 3 se reemplaza por el argumento `metaclass=` en la declaración.

```python
# PYTHON 2 (obsoleto) - NO USAR
class MiClase:
    __metaclass__ = MiMetaclase

# PYTHON 3 (correcto)
class MiClase(metaclass=MiMetaclase):
    pass
```

### Metaclase básica

Una metaclase hereda de `type` y sobrescribe `__new__` o `__init__`.

In [ ]:
class MetaBasica(type):
    """Metaclase simple que imprime al crear clases."""
    def __new__(mcs, nombre, bases, namespace):
        print(f"  [MetaBasica.__new__] Creando clase '{nombre}'")
        return super().__new__(mcs, nombre, bases, namespace)


class Base(metaclass=MetaBasica):
    pass

class SubA(Base):
    pass

class SubB(Base):
    pass

In [ ]:
# Metaclase con registro automático
class Registry(type):
    """Registra automáticamente todas las clases que se crean."""
    _clases: dict[str, type] = {}

    def __new__(mcs, nombre, bases, namespace):
        cls = super().__new__(mcs, nombre, bases, namespace)
        if bases:  # no registrar la clase raíz
            Registry._clases[nombre] = cls
        return cls

    @classmethod
    def obtener_clases(mcs) -> dict[str, type]:
        return dict(mcs._clases)


class PluginBase(metaclass=Registry):
    """Clase base para plugins. Todas las subclases se registran automáticamente."""
    pass


class PluginEmail(PluginBase):
    def ejecutar(self) -> str:
        return "Enviando email"


class PluginSMS(PluginBase):
    def ejecutar(self) -> str:
        return "Enviando SMS"


print("Clases registradas:")
for nombre, cls in Registry.obtener_clases().items():
    print(f"  {nombre}: {cls}")

### `__new__` vs `__init__` en metaclases

| Método | Cuándo se ejecuta | Para qué |
|--------|-------------------|----------|
| `__new__(mcs, nombre, bases, namespace)` | **Antes** de que la clase exista | Crear/modificar la clase, filtrar atributos |
| `__init__(cls, nombre, bases, namespace)` | **Después** de que `__new__` retorna la clase | Configurar la clase, registrar, validar |

In [ ]:
class MetaConInit(type):
    """Demostración de __new__ vs __init__ en metaclases."""

    def __new__(mcs, nombre, bases, namespace):
        print(f"\n1. __new__('{nombre}') - la clase aún no existe")
        cls = super().__new__(mcs, nombre, bases, namespace)
        print(f"   Clase creada: {cls}")
        return cls

    def __init__(cls, nombre, bases, namespace):
        print(f"2. __init__('{nombre}') - la clase ya existe")
        super().__init__(nombre, bases, namespace)
        print(f"   cls es ahora: {cls}")


class Animal(metaclass=MetaConInit):
    pass

class Perro(Animal):
    pass

---
## 6. `__init_subclass__` y hooks

`__init_subclass__` es un hook que se ejecuta **automáticamente** cada vez que se define una subclase. Es la alternativa práctica más simple a las metaclases para muchos casos.

```ascii
  class Padre:
      def __init_subclass__(cls, **kwargs):
          # Se ejecuta al definir cada subclase
          pass

  class Hijo(Padre):  # ← __init_subclass__ se invoca aquí
      pass
```

### Registry pattern con `__init_subclass__`

Registro automático de subclases sin metaclases.

In [ ]:
class Servicio:
    """Clase base que registra automáticamente sus subclases."""
    _registry: dict[str, type["Servicio"]] = {}

    def __init_subclass__(cls, alias: str = "", **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        registro = alias or cls.__name__.lower()
        Servicio._registry[registro] = cls
        print(f"  Servicio registrado: '{registro}' -> {cls.__name__}")

    @classmethod
    def obtener(clase, alias: str) -> type["Servicio"]:
        if alias not in clase._registry:
            raise ValueError(
                f"Servicio '{alias}' no encontrado. "
                f"Disponibles: {list(clase._registry.keys())}"
            )
        return clase._registry[alias]

    @classmethod
    def listar(clase) -> list[str]:
        return list(clase._registry.keys())


class ServicioEmail(Servicio, alias="email"):
    def enviar(self, destino: str, mensaje: str) -> str:
        return f"Email a {destino}: {mensaje}"


class ServicioSMS(Servicio, alias="sms"):
    def enviar(self, destino: str, mensaje: str) -> str:
        return f"SMS a {destino}: {mensaje}"


class ServicioPush(Servicio):  # alias por defecto: "serviciopush"
    def enviar(self, destino: str, mensaje: str) -> str:
        return f"Push a {destino}: {mensaje}"


print(f"\nServicios disponibles: {Servicio.listar()}")

# Obtener e instanciar por alias
cls_email = Servicio.obtener("email")
servicio = cls_email()
print(servicio.enviar("ana@email.com", "Hola!")
)

### ¿Cuándo NO usar metaclases? (YAGNI)

En la mayoría de los casos, `__init_subclass__` o decoradores son suficientes. Las metaclases son el último recurso.

```ascii
  Complejidad creciente:

  Decorador          ────►  __init_subclass__  ────►  Metaclase
  (simple)                  (intermedio)              (complejo)

  ¿Necesitas interceptar    ¿Necesitas registrar     ¿Necesitas modificar
  la definición de una      subclases o inyectar     el namespace de la
  función/método?           comportamiento base?     clase al CREARSE?
```

---
## 7. ABC y `abstractmethod`

`abc.ABC` y `@abstractmethod` permiten definir **interfaces** que las subclases deben implementar obligatoriamente.

In [ ]:
from abc import ABC, abstractmethod


class FormaGeometrica(ABC):
    """Interfaz abstracta para formas geométricas."""

    @abstractmethod
    def area(self) -> float:
        """Calcula el área de la forma."""
        ...

    @abstractmethod
    def perimetro(self) -> float:
        """Calcula el perímetro de la forma."""
        ...

    def describir(self) -> str:
        """Método concreto: no es abstracto, tiene implementación."""
        return (
            f"{type(self).__name__}: "
            f"área={self.area():.2f}, perímetro={self.perimetro():.2f}"
        )


# Intentar instanciar directamente
try:
    forma = FormaGeometrica()
except TypeError as e:
    print(f"Error al instanciar: {e}")

In [ ]:
import math


class Circulo(FormaGeometrica):
    def __init__(self, radio: float) -> None:
        self.radio = radio

    def area(self) -> float:
        return math.pi * self.radio ** 2

    def perimetro(self) -> float:
        return 2 * math.pi * self.radio


class Rectangulo(FormaGeometrica):
    def __init__(self, base: float, altura: float) -> None:
        self.base = base
        self.altura = altura

    def area(self) -> float:
        return self.base * self.altura

    def perimetro(self) -> float:
        return 2 * (self.base + self.altura)


class Triangulo(FormaGeometrica):
    def __init__(self, a: float, b: float, c: float) -> None:
        self.a, self.b, self.c = a, b, c

    def area(self) -> float:
        # Fórmula de Herón
        s = (self.a + self.b + self.c) / 2
        return math.sqrt(s * (s - self.a) * (s - self.b) * (s - self.c))

    def perimetro(self) -> float:
        return self.a + self.b + self.c


# Subclase incompleta - error al instanciar
class Trapecio(FormaGeometrica):
    def area(self) -> float:
        return 0.0
    # Falta implementar perimetro()


try:
    t = Trapecio()
except TypeError as e:
    print(f"Trapecio incompleto: {e}")

# Uso correcto
figuras = [Circulo(5), Rectangulo(4, 6), Triangulo(3, 4, 5)]
print("\nFiguras geométricas:")
for f in figuras:
    print(f"  {f.describir()}")

---
## 8. Advertencias y mejores prácticas

### ¿Cuándo NO usar metaprogramación?

```
  ┌─────────────────────────────────────────────────────────────────┐
  │                   REGLA DE ORO                                  │
  │                                                                 │
  │  "Use la alternativa más simple que funcione."                  │
  │                                                                 │
  │  1. Función normal     → ¿Funciona? SÍ → No uses decorador     │
  │  2. Decorador          → ¿Funciona? SÍ → No uses __init_sub__  │
  │  3. __init_subclass__  → ¿Funciona? SÍ → No uses metaclass     │
  │  4. Metaclase          → Solo si todo lo anterior es insuficiente│
  └─────────────────────────────────────────────────────────────────┘
```

### Problemas comunes con metaprogramación

| Problema | Causa | Solución |
|----------|-------|----------|
| **Debug difícil** | La lógica está "escondida" en metaclases/descriptores | Documentar exhaustivamente |
| **IDE no entiende** | Auto-completado no resuelve dinamismo | Usar type hints explícitos |
| **Conflictos entre metaclases** | Múltiples metaclases incompatibles | Crear metaclase combinada |
| **Rendimiento** | Resolución de atributos más lenta | Medir antes de optimizar |
| **Legibilidad** | Código "mágico" para otros desarrolladores | Comentar la intención |

### Buenas prácticas

```python
# MAL: Metaclase innecesaria
class ValidarMeta(type):
    def __new__(mcs, nombre, bases, namespace):
        # Validar que todos los métodos tengan docstring
        for key, val in namespace.items():
            if callable(val) and not key.startswith('_'):
                if not val.__doc__:
                    raise ValueError(f'{nombre}.{key} necesita docstring')
        return super().__new__(mcs, nombre, bases, namespace)

# BIEN: Decorador simple (más legible)
def requiere_docstring(func):
    if not func.__doc__:
        raise ValueError(f'{func.__name__} necesita docstring')
    return func
```

---
## Diagrama de Referencia: Cadena type → class → instance

```
  ╔═══════════════════════════════════════════════════════════════════╗
  ║              CADENA COMPLETA: type → class → instance            ║
  ╠═══════════════════════════════════════════════════════════════════╣
  ║                                                                   ║
  ║  type (meta-metaclase, es instancia de sí mismo)                  ║
  ║    │                                                              ║
  ║    │ type.__call__() se ejecuta al crear una clase                ║
  ║    ▼                                                              ║
  ║  ┌──────────────┐                                                 ║
  ║  │ MiMetaclase  │  type.__new__(MiMetaclase, ...)                 ║
  ║  └──────┬───────┘                                                 ║
  ║         │ MiMetaclase.__call__() se ejecuta al crear instancias   ║
  ║         ▼                                                         ║
  ║  ┌──────────────┐                                                 ║
  ║  │   MiClase    │  MiMetaclase.__new__(MiClase, ...)              ║
  ║  │              │  → Descriptor.__set_name__(MiClase, 'nombre')   ║
  ║  └──────┬───────┘                                                 ║
  ║         │ MiClase.__call__() se ejecuta al hacer MiClase()        ║
  ║         ▼                                                         ║
  ║  ┌──────────────┐                                                 ║
  ║  │  instancia   │  MiClase.__new__(MiClase)                      ║
  ║  │              │  MiClase.__init__(instancia)                    ║
  ║  └──────────────┘                                                 ║
  ║                                                                   ║
  ║  Flujo de acceso a atributo:                                     ║
  ║  instancia.nombre                                                 ║
  ║    1. Busca en type(instancia).__mro__ (descriptores data)        ║
  ║    2. Busca en instancia.__dict__                                 ║
  ║    3. Busca en type(instancia).__mro__ (descriptores non-data)    ║
  ║                                                                   ║
  ╚═══════════════════════════════════════════════════════════════════╝
```

### Diagrama de Dataflow: Descriptor en acción

```
  ╔══════════════════════════════════════════════════════════════╗
  ║           DATAFLOW: Descriptor interceptando acceso         ║
  ╠══════════════════════════════════════════════════════════════╣
  ║                                                              ║
  ║  INSTANCIA obj               CLASE MiClase                  ║
  ║  ┌──────────┐               ┌──────────────────┐            ║
  ║  │ __dict__:│               │ MiDescriptor:    │            ║
  ║  │  _x: 10  │               │  __get__(...)    │            ║
  ║  └──────────┘               │  __set__(...)    │            ║
  ║       │                     │  __set_name__(..)│            ║
  ║       │                     └────────┬─────────┘            ║
  ║       │                              │                      ║
  ║       │   obj.x = 5                  │                      ║
  ║       └──────────────────────────────┼───────────────┐      ║
  ║                                      │               │      ║
  ║                                      ▼               │      ║
  ║                              ┌──────────────┐        │      ║
  ║                              │__set__(obj,  │        │      ║
  ║                              │     value=5) │        │      ║
  ║                              └──────┬───────┘        │      ║
  ║                                     │                │      ║
  ║                                     ▼                │      ║
  ║                              ┌──────────────┐        │      ║
  ║                              │ Valida tipo  │        │      ║
  ║                              │ Almacena en  │        │      ║
  ║                              │ __dict__     │        │      ║
  ║                              └──────────────┘        │      ║
  ║                                                      │      ║
  ║   obj.x  ◄──────────────────────────────────────────┘      ║
  ║     │                                                        ║
  ║     ▼                                                        ║
  ║  ┌──────────────┐                                            ║
  ║  │__get__(obj)  │                                            ║
  ║  │ → retorna 5  │                                            ║
  ║  └──────────────┘                                            ║
  ║                                                              ║
  ╚══════════════════════════════════════════════════════════════╝
```

---
## Tabla de Referencia Rápida

| Concepto | Qué es | Cuándo usarlo |
|----------|--------|---------------|
| **`type(obj)`** | Retorna el tipo de un objeto | Inspección, debugging |
| **`type(name, bases, dict)`** | Crea una clase dinámicamente | Frameworks, ORM, serialización |
| **Descriptor** | Objeto con `__get__`/`__set__`/`__delete__` | Control de acceso, validación |
| **Data descriptor** | Tiene `__get__` + `__set__` | Cuando debes interceptar escrituras |
| **Non-data descriptor** | Solo tiene `__get__` | Métodos, valores computados |
| **`__set_name__`** | Descriptor que conoce su nombre | Descriptores reutilizables |
| **Metaclase** | Clase de una clase (hereda de `type`) | Modificar creación de clases |
| **`__init_subclass__`** | Hook al definir subclase | Registry, validación, decorar |
| **ABC + abstractmethod** | Interfaz obligatoria | Contratos, plugins |
| **`metaclass=`** | Sintaxis para asignar metaclase | Python 3 (reemplaza `__metaclass__`) |

---
## Ejercicios

### Ejercicio Guiado 1: Descriptor de validación de rango

Crea un descriptor `RangoValido` que valide que un valor numérico esté dentro de un rango `[min, max]` al asignar.

In [ ]:
class RangoValido:
    """Descriptor que valida que un valor esté dentro de un rango."""

    def __init__(self, minimo: float, maximo: float) -> None:
        if minimo > maximo:
            raise ValueError(f"minimo ({minimo}) no puede ser mayor que maximo ({maximo})")
        self.minimo = minimo
        self.maximo = maximo
        self.name = ""

    def __set_name__(self, owner, name) -> None:
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(f"_rango_{self.name}")

    def __set__(self, obj, value) -> None:
        if not isinstance(value, (int, float)):
            raise TypeError(f"'{self.name}' debe ser numérico, recibió {type(value).__name__}")
        if not (self.minimo <= value <= self.maximo):
            raise ValueError(
                f"'{self.name}' = {value} está fuera del rango "
                f"[{self.minimo}, {self.maximo}]"
            )
        obj.__dict__[f"_rango_{self.name}"] = value


class Sensor:
    temperatura = RangoValido(-50, 150)
    humedad = RangoValido(0, 100)
    presion = RangoValido(900, 1100)

    def __init__(self, temp: float, hum: float, pres: float) -> None:
        self.temperatura = temp
        self.humedad = hum
        self.presion = pres

    def __repr__(self) -> str:
        return (
            f"Sensor(temp={self.temperatura}, "
            f"hum={self.humedad}, pres={self.presion})"
        )


s = Sensor(25.5, 60, 1013)
print(s)

# Valores válidos
s.temperatura = 36.6
s.humedad = 45
print(f"Actualizado: {s}")

# Valor inválido - fuera de rango
try:
    s.humedad = 150
except ValueError as e:
    print(f"Error: {e}")

# Tipo inválido
try:
    s.presion = "alta"
except TypeError as e:
    print(f"Error: {e}")

### Ejercicio Guiado 2: Metaclase que añade `__repr__` automáticamente
Crea una metaclase que, al definir una clase, genere automáticamente un `__repr__` basado en los atributos de instancia definidos en `__init__`.

In [ ]:
import inspect


class AutoRepr(type):
    """Metaclase que genera __repr__ automáticamente."""

    def __new__(mcs, nombre, bases, namespace):
        # Solo generar __repr__ si no está definido y no es la clase base
        if "__repr__" not in namespace and bases:
            # Intentar extraer nombres de parámetros de __init__
            init_method = namespace.get("__init__")
            if init_method:
                sig = inspect.signature(init_method)
                params = [
                    p.name for p in sig.parameters.values()
                    if p.name != "self"
                ]

                def make_repr(attrs):
                    def __repr__(self) -> str:
                        campos = ", ".join(
                            f"{a}={getattr(self, a)!r}" for a in attrs
                        )
                        return f"{type(self).__name__}({campos})"
                    return __repr__

                namespace["__repr__"] = make_repr(params)

        return super().__new__(mcs, nombre, bases, namespace)


class Punto(metaclass=AutoRepr):
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y


class Rectangulo(metaclass=AutoRepr):
    def __init__(self, base: float, altura: float, color: str = "rojo") -> None:
        self.base = base
        self.altura = altura
        self.color = color


p = Punto(3, 4)
r = Rectangulo(10, 5, "azul")

print(f"Punto:     {p}")
print(f"Rectángulo: {r}")
print(f"\n__repr__ generado por AutoRepr:")
print(f"  Punto.__repr__ = {Punto.__repr__}")

### Ejercicio Guiado 3: ABC con validación de estado

Crea una interfaz `EstadoTransicionable` que obligue a las subclases a definir transiciones válidas y un estado inicial.

In [ ]:
from abc import ABC, abstractmethod


class EstadoTransicionable(ABC):
    """Interfaz para objetos con estados y transiciones."""

    def __init__(self) -> None:
        self._estado: str = self.estado_inicial()

    @abstractmethod
    def estado_inicial(self) -> str:
        """Retorna el estado inicial."""
        ...

    @abstractmethod
    def transiciones_validas(self) -> dict[str, list[str]]:
        """Retorna un dict {estado: [estados_posibles]}"""
        ...

    @property
    def estado(self) -> str:
        return self._estado

    def transicionar(self, nuevo_estado: str) -> str:
        """Intenta transicionar a un nuevo estado."""
        validas = self.transiciones_validas().get(self._estado, [])
        if nuevo_estado not in validas:
            raise ValueError(
                f"Transición inválida: '{self._estado}' → '{nuevo_estado}'. "
                f"Estados válidos desde '{self._estado}': {validas}"
            )
        anterior = self._estado
        self._estado = nuevo_estado
        return f"{anterior} → {nuevo_estado}"


class Pedido(EstadoTransicionable):

    def estado_inicial(self) -> str:
        return "pendiente"

    def transiciones_validas(self) -> dict[str, list[str]]:
        return {
            "pendiente": ["confirmado", "cancelado"],
            "confirmado": ["enviado", "cancelado"],
            "enviado": ["entregado"],
            "entregado": [],
            "cancelado": [],
        }


ped = Pedido()
print(f"Estado inicial: {ped.estado}")

print(f"Transición 1: {ped.transicionar('confirmado')}")
print(f"Transición 2: {ped.transicionar('enviado')}")
print(f"Transición 3: {ped.transicionar('entregado')}")

# Transición inválida
try:
    ped.transicionar('pendiente')  # no se puede volver atrás
except ValueError as e:
    print(f"\nError: {e}")

# Intentar instanciar la clase abstracta
try:
    e = EstadoTransicionable()
except TypeError as e:
    print(f"\nError al instanciar abstracta: {e}")

### Ejercicio Independiente

Crea un **framework de validación de atributos** combinando descriptores con un registry usando `__init_subclass__`.

**Requisitos:**
1. Un descriptor `Validado` que acepte un tipo y una función de validación.
2. Una clase base `ModeloValidado` que registre automáticamente todas las subclases.
3. Un método estático `ModeloValidado.validar_instancia(obj)` que verifique que todos los atributos validados cumplan las reglas.
4. Crear al menos 2 modelos (ej: `Usuario`, `Producto`) que usen el framework.

**Pista:** El descriptor debe usar `__set_name__` para conocer su nombre y generar warnings al acceder atributos no validados.

In [ ]:
class Validado:
    """Descriptor que valida tipo y condición personalizada."""

    def __init__(self, tipo: type, validacion=None, mensaje: str = "") -> None:
        self.tipo = tipo
        self.validacion = validacion
        self.mensaje = mensaje or f"Valor no válido para {tipo.__name__}"
        self.name = ""

    def __set_name__(self, owner, name) -> None:
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(f"_val_{self.name}")

    def __set__(self, obj, value) -> None:
        if not isinstance(value, self.tipo):
            raise TypeError(
                f"'{self.name}' debe ser {self.tipo.__name__}, "
                f"recibió {type(value).__name__}"
            )
        if self.validacion and not self.validacion(value):
            raise ValueError(self.mensaje)
        obj.__dict__[f"_val_{self.name}"] = value


class ModeloValidado:
    """Clase base que registra subclases y sus campos validados."""
    _modelos: dict[str, type] = {}

    def __init_subclass__(cls, **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        ModeloValidado._modelos[cls.__name__] = cls

    @staticmethod
    def validar_instancia(obj) -> list[str]:
        """Valida todos los campos validados de una instancia."""
        errores: list[str] = []
        for key in dir(type(obj)):
            attr = getattr(type(obj), key, None)
            if isinstance(attr, Validado):
                valor = getattr(obj, key, None)
                if valor is None:
                    errores.append(f"'{key}' no tiene valor asignado")
        return errores

    @classmethod
    def listar_modelos(clase) -> list[str]:
        return list(ModeloValidado._modelos.keys())


# --- Crear modelos ---

class Usuario(ModeloValidado):
    nombre = Validado(str, lambda v: len(v) >= 2, "Nombre debe tener 2+ caracteres")
    email = Validado(str, lambda v: "@" in v, "Email debe contener @")
    edad = Validado(int, lambda v: 0 <= v <= 150, "Edad debe estar entre 0 y 150")

    def __init__(self, nombre: str, email: str, edad: int) -> None:
        self.nombre = nombre
        self.email = email
        self.edad = edad


class Producto(ModeloValidado):
    nombre = Validado(str, lambda v: len(v) > 0, "Nombre no puede estar vacío")
    precio = Validado(float, lambda v: v > 0, "Precio debe ser positivo")
    stock = Validado(int, lambda v: v >= 0, "Stock no puede ser negativo")

    def __init__(self, nombre: str, precio: float, stock: int) -> None:
        self.nombre = nombre
        self.precio = precio
        self.stock = stock


# --- Demostración ---
print(f"Modelos registrados: {ModeloValidado.listar_modelos()}")

u = Usuario("Ana", "ana@mail.com", 30)
errores = ModeloValidado.validar_instancia(u)
print(f"\nValidación Usuario válido: {errores if errores else 'OK'}")

# Valores inválidos
try:
    u.edad = -5
except ValueError as e:
    print(f"Error validación: {e}")

try:
    p = Producto("", -10, 5)
except ValueError as e:
    print(f"Error validación: {e}")

---
## Resumen

| Concepto clave | Idea esencial |
|----------------|---------------|
| **Clases son objetos** | Todo en Python es objeto; las clases son instancias de `type` |
| **Descriptores** | Objetos que interceptan acceso a atributos (`__get__`, `__set__`, `__delete__`) |
| **Data vs non-data** | Data descriptors ganan sobre `__dict__`; non-data pierden |
| **`__set_name__`** | Descriptor auto-conoce su nombre, evita repetición |
| **Metaclases** | Controlan cómo se crean las clases; heredan de `type` |
| **`__init_subclass__`** | Hook alternativo a metaclases para registrar/validar subclases |
| **ABC + abstractmethod** | Forzan implementación de interfaces, rechazan instanciación directa |
| **Regla de oro** | Siempre usar la alternativa más simple que resuelva el problema |

### Cadenas de resolución de Python

```
  Acceso: obj.atributo
  1. Data descriptor en type(obj).__mro__  → GANA
  2. obj.__dict__['atributo']              → si existe
  3. Non-data descriptor en type(obj).__mro__
  4. TypeError: AttributeError
```